In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/predict-the-floodd/sample_submission.csv
/kaggle/input/predict-the-floodd/train.csv
/kaggle/input/predict-the-floodd/test.csv


import numpy as np
import pandas as pd

def create_advanced_sum_features(df, risk_features):
    # 1. Base Sum (The Golden Feature)
    df['fsum'] = df[risk_features].sum(axis=1)
    df['fstd'] = df[risk_features].std(axis=1)
    df['fmean'] = df[risk_features].mean(axis=1)
    df['fkurtosis'] = df[risk_features].kurtosis(axis=1)

    
    # 2. Count of "1s" (Complexity/Variance proxy)
    df['one_count'] = (df[risk_features] == 1).sum(axis=1)
    
    # 3. Polynomials
    df['fsum2'] = df['fsum'] ** 2
    
    # 4. Logs (Safe to do on the sum)
    # We add +1 to avoid log(0) errors just in case
    df['log_sum'] = np.log1p(df['fsum'])
    df['log2_sum'] = np.log2(df['fsum'] + 1)
    
    # 5. Exponentials (The Dangerous Part - Handling Safely)
    # Instead of exp(TotalSum), we do Sum(exp(features)) to avoid overflow
    # Or we normalize the sum first. Here is the safest variant:
    
    # Variant A: Sum of Exponentials (captures high-value outliers in rows)
    # Scale inputs by 0.1 to keep exponents manageable
    df['exp_sum'] = np.exp(df[risk_features] * 0.1).sum(axis=1) 
    df['exp2_sum'] = np.exp(df[risk_features] * 0.2).sum(axis=1)
    df['exp3_sum'] = np.exp(df[risk_features] * 0.3).sum(axis=1)

    sorted_features = np.sort(df[risk_features].values, axis=1)
    for i in range(sorted_features.shape[1]):
        df[f'sort_{i}'] = sorted_features[:, i]
        
    # 4. Unique Count (Complexity)
    df['unique_count'] = df[risk_features].apply(lambda x: len(np.unique(x)), axis=1)
    
    return df

# --- Execution ---
# ensure risk_features does NOT contain 'id', 'target', or 'FloodProbability'
# risk_features = [ ... your 20 columns ... ]

train_df = create_advanced_sum_features(train_data.copy(), train_data.columns.difference(['FloodProbability']))
test_df = create_advanced_sum_features(test_data.copy(), train_data.columns.difference(['FloodProbability']))

def create_robust_features(df, features):
    df['f_median'] = df[features].median(axis=1)
    quantile_25 = df[features].quantile(0.25, axis=1)
    quantile_75 = df[features].quantile(0.75, axis=1)
    df['f_iqr_sum'] = df[features].apply(
        lambda x: x[(x >= x.quantile(0.25)) & (x <= x.quantile(0.75))].sum(), 
        axis=1
    )
    df['f_iqr_range'] = quantile_75 - quantile_25
    df['f_trimean'] = (quantile_25 + (2 * df['f_median']) + quantile_75) / 4

    return df
print("Trimean Correlation:", train_df['f_median'].corr(train_df['FloodProbability']))

df['golden_sum'] = sorted_data[:, 0:15].sum(axis=1)
df['peak_mean'] = sorted_data[:, 5:10].mean(axis=1)
df['tail_sum'] = sorted_data[:, 15:].sum(axis=1)
df['risk_consistency'] = df['golden_sum'] / (df['tail_sum'] + 1)

In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')


df_train = pd.read_csv('/kaggle/input/predict-the-floodd/train.csv')
df_test = pd.read_csv('/kaggle/input/predict-the-floodd/test.csv')

if 'id' in df_train.columns:
    df_train = df_train.drop(columns=['id'])
if 'id' in df_test.columns:
    submission_ids = df_test['id']
    df_test = df_test.drop(columns=['id'])

X = df_train.drop(columns=['FloodProbability'])
y = df_train['FloodProbability']
X_test = df_test


def enhance_features(df,features):
   
    df['sum_risk'] = df.sum(axis=1)
    df['mean_risk'] = df.mean(axis=1)
    df['std_risk'] = df.std(axis=1)
    df['max_risk'] = df.max(axis=1)
    df['min_risk'] = df.min(axis=1)
    df['median_risk'] = df.median(axis=1)
    df['f_median'] = df[features].median(axis=1)
    quantile_25 = df[features].quantile(0.25, axis=1)
    quantile_75 = df[features].quantile(0.75, axis=1)
    df['f_iqr_sum'] = df[features].apply(
        lambda x: x[(x >= x.quantile(0.25)) & (x <= x.quantile(0.75))].sum(), 
        axis=1
    )
    sorted_data = np.sort(df[features].values, axis=1)
    df['golden_sum'] = sorted_data[:, 0:15].sum(axis=1)
    df['peak_mean'] = sorted_data[:, 5:10].mean(axis=1)
    df['tail_sum'] = sorted_data[:, 15:].sum(axis=1)
    df['risk_consistency'] = df['golden_sum'] / (df['tail_sum'] + 1)
    df['f_iqr_range'] = quantile_75 - quantile_25
    df['f_trimean'] = (quantile_25 + (2 * df['f_median']) + quantile_75) / 4

    # Interaction: Special characters of the synthetic generator 
    # often respond to unique value counts per row
    df['unique_values'] = df.apply(lambda x: len(np.unique(x)), axis=1)
    
    return df

print("Engineering features...")
X = enhance_features(X,X.columns.difference(['FloodProbability']))
X_test = enhance_features(X_test,X_test.columns.difference(['id']))


scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)



xgb_params = {
    'n_estimators': 2000,
    'learning_rate': 0.01,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'n_jobs': -1,
    'random_state': 42,
    'tree_method': 'hist' # Faster training
}

lgb_params = {
    'n_estimators': 2000,
    'learning_rate': 0.01,
    'max_depth': -1,
    'num_leaves': 31,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'n_jobs': -1,
    'random_state': 42,
    'verbosity': -1
}

cat_params = {
    'iterations': 2000,
    'learning_rate': 0.01,
    'depth': 6,
    'subsample': 0.8,
    'verbose': 0,
    'random_seed': 42,
    'allow_writing_files': False
}

n_splits = 10
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_preds = np.zeros(X.shape[0])
test_preds = np.zeros(X_test.shape[0])
scores = []

print(f"Starting {n_splits}-Fold Cross-Validation...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_scaled, y)):
    X_train_f, X_val_f = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
    y_train_f, y_val_f = y.iloc[train_idx], y.iloc[val_idx]


    model_xgb = xgb.XGBRegressor(**xgb_params)
    model_xgb.fit(X_train_f, y_train_f, 
                  eval_set=[(X_val_f, y_val_f)], 
                  verbose=False)
    val_xgb = model_xgb.predict(X_val_f)
    test_xgb = model_xgb.predict(X_test_scaled)

    model_lgb = lgb.LGBMRegressor(**lgb_params)
    model_lgb.fit(X_train_f, y_train_f, 
                  eval_set=[(X_val_f, y_val_f)], 
                  eval_metric='r2',
                  callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)])
    val_lgb = model_lgb.predict(X_val_f)
    test_lgb = model_lgb.predict(X_test_scaled)


    model_cat = CatBoostRegressor(**cat_params)
    model_cat.fit(X_train_f, y_train_f, 
                  eval_set=(X_val_f, y_val_f), 
                  early_stopping_rounds=100)
    val_cat = model_cat.predict(X_val_f)
    test_cat = model_cat.predict(X_test_scaled)


    val_pred = (0.34 * val_xgb) + (0.33 * val_lgb) + (0.33 * val_cat)
    test_pred = (0.34 * test_xgb) + (0.33 * test_lgb) + (0.33 * test_cat)

    oof_preds[val_idx] = val_pred
    test_preds += test_pred / n_splits
    
    score = r2_score(y_val_f, val_pred)
    scores.append(score)
    print(f"Fold {fold+1} R2 Score: {score:.5f}")


print("-" * 30)
print(f"Mean R2 Score: {np.mean(scores):.5f}")
print("-" * 30)


submission = pd.DataFrame({'id': submission_ids, 'FloodProbability': test_preds})
submission.to_csv('submission_ensemble_engineered.csv', index=False)
print("Submission file 'submission_ensemble.csv' created.")

Engineering features...
Starting 10-Fold Cross-Validation...
Fold 1 R2 Score: 0.71898
Fold 2 R2 Score: 0.72356
Fold 3 R2 Score: 0.72506
Fold 4 R2 Score: 0.72736
Fold 5 R2 Score: 0.71717
Fold 6 R2 Score: 0.71795
Fold 7 R2 Score: 0.72549
Fold 8 R2 Score: 0.72071
Fold 9 R2 Score: 0.72507
Fold 10 R2 Score: 0.71220
------------------------------
Mean R2 Score: 0.72135
------------------------------
Submission file 'submission_ensemble.csv' created.
